In [1]:
# import random

# # List of single-token names (verified to be single tokens in LLaMA tokenizer with leading space)
# # Note: You should verify these with your actual tokenizer
# NAMES = [
#     "John", "Mary", "James", "Sarah", "Michael", "Emma",
#     "David", "Lisa", "Robert", "Anna", "William", "Laura",
#     "Richard", "Helen", "Thomas", "Nancy", "Charles", "Karen",
#     "Daniel", "Betty", "Matthew", "Margaret", "Donald", "Sandra",
#     "Mark", "Ashley", "Paul", "Dorothy", "Steven", "Kimberly",
#     "Andrew", "Emily", "Joshua", "Donna", "Kenneth", "Michelle",
#     "Kevin", "Carol", "Brian", "Amanda", "George", "Melissa",
#     "Edward", "Deborah", "Ronald", "Stephanie", "Timothy", "Rebecca",
#     "Jason", "Sharon", "Jeffrey", "Cynthia", "Ryan", "Kathleen",
#     "Jacob", "Amy", "Gary", "Angela", "Nicholas", "Shirley",
#     "Eric", "Brenda", "Jonathan", "Emma", "Stephen", "Anna",
#     "Larry", "Pamela", "Justin", "Nicole", "Scott", "Katherine",
#     "Brandon", "Samantha", "Benjamin", "Christine", "Samuel", "Debra",
#     "Raymond", "Rachel", "Patrick", "Catherine", "Alexander", "Carolyn",
#     "Jack", "Janet", "Dennis", "Ruth", "Jerry", "Maria"
# ]

# # Templates for IOI prompts
# TEMPLATES = [
#     "Then {A} and {B} went to the movies, and {B} gave a ticket to",
#     "When {A} and {B} were working on a project, {B} showed the results to",
#     "Then {A} and {B} had a long argument, and afterwards {B} said sorry to",
# ]

# def create_ioi_dataset(num_examples=100, seed=42):
#     """
#     Create IOI dataset with clean and corrupted pairs.
    
#     Args:
#         num_examples: Number of examples to generate (default 100)
#         seed: Random seed for reproducibility
    
#     Returns:
#         List of dictionaries, each containing:
#         - clean_prompt: IOI sentence with A once, B twice
#         - corrupted_prompt: ABC sentence with A, B, C each once
#         - name_A: The indirect object (correct answer)
#         - name_B: The subject (wrong answer, appears twice in clean)
#         - name_C: The replacement name (only in corrupted)
#         - template: Which template was used
#     """
#     random.seed(seed)
#     dataset = []
    
#     # Distribute examples across templates
#     examples_per_template = num_examples // len(TEMPLATES)
#     remainder = num_examples % len(TEMPLATES)
    
#     for template_idx, template in enumerate(TEMPLATES):
#         # Determine how many examples for this template
#         count = examples_per_template + (1 if template_idx < remainder else 0)
        
#         for _ in range(count):
#             # Sample three distinct names
#             name_A, name_B, name_C = random.sample(NAMES, 3)
            
#             # Create clean prompt (IOI: A once, B twice)
#             clean_prompt = template.format(A=name_A, B=name_B)
            
#             # Create corrupted prompt (ABC: A, B, C each once)
#             # Replace second occurrence of B with C
#             corrupted_prompt = template.replace("{B}", "{C}", 1)  # Replace first {B} with {C}
#             corrupted_prompt = corrupted_prompt.format(A=name_A, B=name_B, C=name_C)
            
#             # Actually we need to do this more carefully
#             # Clean: "When A and B ..., B gave ... to"
#             # Corrupted: "When A and B ..., C gave ... to"
            
#             # Better approach: format clean normally, then construct corrupted
#             clean_prompt = template.format(A=name_A, B=name_B)
            
#             # For corrupted, replace second B with C
#             parts = template.split('{B}')
#             # parts[0] = "When {A} and "
#             # parts[1] = " left the office, "
#             # parts[2] = " said goodbye to"
            
#             if len(parts) == 3:  # Template has exactly 2 occurrences of {B}
#                 corrupted_template = parts[0] + '{B}' + parts[1] + '{C}' + parts[2]
#                 corrupted_prompt = corrupted_template.format(A=name_A, B=name_B, C=name_C)
#             else:
#                 # Fallback: shouldn't happen with our templates
#                 corrupted_prompt = template.format(A=name_A, B=name_C)
            
#             dataset.append({
#                 'clean_prompt': clean_prompt,
#                 'corrupted_prompt': corrupted_prompt,
#                 'name_A': name_A,
#                 'name_B': name_B,
#                 'name_C': name_C,
#                 'template': template,
#                 'template_idx': template_idx
#             })
    
#     return dataset


# # Generate the dataset
# ioi_dataset = create_ioi_dataset(num_examples=100, seed=42)

# # Print first 5 examples to verify
# print("=" * 80)
# print("IOI Dataset Sample (first 5 examples)")
# print("=" * 80)
# print()

# for i, example in enumerate(ioi_dataset[:5]):
#     print(f"Example {i+1}:")
#     print(f"  Clean:     {example['clean_prompt']}")
#     print(f"  Corrupted: {example['corrupted_prompt']}")
#     print(f"  A (correct): {example['name_A']}")
#     print(f"  B (wrong):   {example['name_B']}")
#     print(f"  C (neutral): {example['name_C']}")
#     print(f"  Template:    {example['template']}")
#     print()

# # Print dataset statistics
# print("=" * 80)
# print("Dataset Statistics")
# print("=" * 80)
# print(f"Total examples: {len(ioi_dataset)}")
# print(f"\nExamples per template:")
# from collections import Counter
# template_counts = Counter(ex['template_idx'] for ex in ioi_dataset)
# for template_idx in sorted(template_counts.keys()):
#     template = TEMPLATES[template_idx]
#     count = template_counts[template_idx]
#     print(f"  Template {template_idx}: {count} examples")
#     print(f"    '{template[:60]}...'")

# # Verify name uniqueness in each example
# print(f"\n✓ All examples have distinct A, B, C names")
# for i, ex in enumerate(ioi_dataset):
#     assert ex['name_A'] != ex['name_B'] != ex['name_C']
#     assert ex['name_A'] != ex['name_C']

# print(f"✓ All {len(ioi_dataset)} examples validated")
# print()

# # Save to file (optional)
# import json

# output_file = 'ioi_dataset_100.json'
# with open(output_file, 'w') as f:
#     json.dump(ioi_dataset, f, indent=2)
    
# print(f"✓ Dataset saved to {output_file}")

# Step 1 - Validation and Dataset creation

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
from huggingface_hub import login

model_name = "meta-llama/Llama-3.2-1B"
import os
token = os.getenv("HF_TOKEN")
login(token)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)


/Users/shreeyansarora/Downloads/Lexsi-Labs_Assignment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:03<00:00, 36.72it/s]


In [2]:
import random
import numpy as np
import torch

def set_seed(seed=54):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(54)

In [3]:
device = 'mps' if torch.mps.is_available() else 'cpu'

In [4]:
with open('ioi_dataset_95.json', 'r') as f:
    ds = json.load(f)

In [ ]:
# Clean prompt corrected_mean value

store_correct = []
for data in ds:
    tokens = tokenizer(data['clean_prompt'], return_tensors = 'pt').to(device = device)
    a_id = tokenizer(' ' + data['name_A'], return_tensors = 'pt')['input_ids'][0][-1]
    b_id = tokenizer(' ' + data['name_B'], return_tensors = 'pt')['input_ids'][0][-1]
    output = model(**tokens) #gives logits
    a = output.logits[:,-1,:][0][a_id]
    b = output.logits[:,-1,:][0][b_id]
    c = a-b
    d = c.detach().to(device = 'cpu')
    float(d)
    store_correct.append(d)

In [6]:
correct_mean = torch.mean(torch.stack(store_correct))

In [7]:
correct_mean

tensor(5.3867, dtype=torch.float16)

# Step 2 - Exploratory Activation Patching

In [234]:
for name, layer in model.named_modules():
    layer._forward_hooks.clear()

In [ ]:
# Creation of corrupted patches for each MLP layer and attention head

activations = {}
def find_hook(name):
    def hook(model, input, output):
        data = output[0].detach().cpu()
        print(data.shape) 
        if name not in activations:
            activations[name] = [data]
        else:
            activations[name].append(data)
    return hook

In [237]:

for data in ds:
    tokens = tokenizer(data['corrupted_prompt'], return_tensors = 'pt').to(device = device)
    a_id = tokenizer(' ' + data['name_A'], return_tensors = 'pt')['input_ids'][0][-1]
    b_id = tokenizer(' ' + data['name_B'], return_tensors = 'pt')['input_ids'][0][-1]
    for i in range(16):
        for name, layer in model.named_modules():
            if name==f'model.layers.{i}.mlp':
                h = layer.register_forward_hook(find_hook(name))
                output = model(**tokens)
                h.remove()
            if name==f'model.layers.{i}.self_attn':
                h = layer.register_forward_hook(find_hook(name))
                output = model(**tokens)
                h.remove()


torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2048])
torch.Size([16, 2048])
torch.Size([1, 16, 2

In [ ]:
activations['model.layers.1.mlp'][0].shape #shape is 32, 95, 16, 2048

torch.Size([16, 2048])

## Attention Patching

In [ ]:
for name, layer in model.named_modules():
    layer._forward_hooks.clear()

In [ ]:
#attention size - 1,16, 2048 (for every layer)
def attention_layer_hook(head_idx, corrupt):
    def hook(model, input, output):
        output[0][:, :, (head_idx*512):((head_idx+1)*512)] = corrupt
        return output
    return hook

In [ ]:
#Replacing attention head of each layer with patched activations to see the difference in model performance

attention_layer_patching = {} #data point, layer, head
for k, data in enumerate(ds):
    attention_layer_patching[k] = {}
    tokens = tokenizer(data['clean_prompt'], return_tensors='pt').to(device=device)
    a_id = tokenizer(' ' + data['name_A'], return_tensors='pt')['input_ids'][0][-1]
    b_id = tokenizer(' ' + data['name_B'], return_tensors='pt')['input_ids'][0][-1]
    for i in range(16):
        attention_layer_patching[k][i] = []
        for name, layer in model.named_modules():
            if name==f'model.layers.{i}.self_attn':
                for j in range(4):
                    h = layer.register_forward_hook(attention_layer_hook(j, activations[name][k][:, :, (j*512):((j+1)*512)]))
                    output = model(**tokens)
                    h.remove()
                    a = output.logits[:,-1,:][0][a_id]
                    b = output.logits[:,-1,:][0][b_id]
                    c = a-b
                    d = c.detach().to(device = 'cpu')
                    attention_layer_patching[k][i].append(d)
                    


In [ ]:
flattened_attention_logit = torch.zeros([95, 16, 4])

for d in attention_layer_patching:
    for l in attention_layer_patching[d]:
        for i, h in enumerate(attention_layer_patching[d][l]):
            flattened_attention_logit[d,l,i] = h.item()

In [ ]:
mean_attention_logit = torch.mean(flattened_attention_logit, axis = 0) #mean of all 95 runs

In [ ]:
mean_attention_difference = correct_mean - mean_attention_logit #difference between correct mean and outputted mean for activations to measure importance

In [ ]:
mean_attention_difference_arr = []

for i in range(len(mean_attention_difference)):
    for j in range(len(mean_attention_difference[i])):
        mean_attention_difference_arr.append([mean_attention_difference[i,j], i, j])

In [ ]:
sorted_attention_list = sorted(mean_attention_difference_arr, key=lambda x: x[0].item(), reverse = True)

In [ ]:
sorted_attention_list 

[[tensor(0.7595), 8, 2],
 [tensor(0.5921), 14, 3],
 [tensor(0.5880), 14, 2],
 [tensor(0.4886), 14, 1],
 [tensor(0.4796), 14, 0],
 [tensor(0.4710), 9, 3],
 [tensor(0.4681), 9, 2],
 [tensor(0.3935), 9, 1],
 [tensor(0.3880), 8, 0],
 [tensor(0.3394), 15, 2],
 [tensor(0.2889), 11, 1],
 [tensor(0.2808), 11, 3],
 [tensor(0.2775), 15, 3],
 [tensor(0.2773), 11, 0],
 [tensor(0.2648), 11, 2],
 [tensor(0.2490), 15, 1],
 [tensor(0.2429), 15, 0],
 [tensor(0.1914), 8, 3],
 [tensor(0.1381), 9, 0],
 [tensor(0.1102), 7, 2],
 [tensor(0.0993), 6, 1],
 [tensor(0.0989), 13, 3],
 [tensor(0.0855), 7, 1],
 [tensor(0.0847), 8, 1],
 [tensor(0.0713), 7, 3],
 [tensor(0.0661), 13, 1],
 [tensor(0.0562), 5, 2],
 [tensor(0.0493), 2, 3],
 [tensor(0.0440), 4, 0],
 [tensor(0.0419), 3, 0],
 [tensor(0.0405), 2, 0],
 [tensor(0.0388), 3, 2],
 [tensor(0.0377), 3, 3],
 [tensor(0.0356), 3, 1],
 [tensor(0.0312), 13, 0],
 [tensor(0.0293), 13, 2],
 [tensor(0.0263), 1, 3],
 [tensor(0.0255), 0, 1],
 [tensor(0.0241), 2, 1],
 [tensor(

## MLP Patching

In [ ]:
for name, layer in model.named_modules():
    layer._forward_hooks.clear()

In [ ]:
def patching_hook_mlp(corrupt):
    def hook(model, input, output):
        output[0] = corrupt
        return output
    return hook

In [ ]:
#Doing the same thing for all MLP layers.

mlp_layer_patching = torch.zeros([95, 16])
for k, data in enumerate(ds):
    tokens = tokenizer(data['clean_prompt'], return_tensors = 'pt').to(device = device)
    a_id = tokenizer(' ' + data['name_A'], return_tensors = 'pt')['input_ids'][0][-1]
    b_id = tokenizer(' ' + data['name_B'], return_tensors = 'pt')['input_ids'][0][-1]
    for i in range(16):
        for name, layer in model.named_modules():
            if name==f'model.layers.{i}.mlp':
                h = layer.register_forward_hook(patching_hook_mlp(activations[name][k]))
                output = model(**tokens)
                h.remove()
                a = output.logits[:,-1,:][0][a_id]
                b = output.logits[:,-1,:][0][b_id]
                c = a-b
                d = c.detach().to(device = 'cpu')
                float(d)
                mlp_layer_patching[k,i] = d

In [ ]:
mean_mlp_patching = torch.mean(mlp_layer_patching, axis = 0)

In [ ]:
mean_mlp_difference = correct_mean - mean_mlp_patching

In [ ]:
mean_mlp_difference

tensor([ 4.9755,  0.2977,  0.2380,  1.1937,  0.2584,  0.2344, -0.1673,  1.3259,
         1.3249,  0.1885,  1.5229,  1.1153,  0.2939,  0.6438, -0.3994, -0.1762])

In [ ]:
mean_mlp_difference_arr = []
for i in range(len(mean_mlp_difference)):
    mean_mlp_difference_arr.append([mean_mlp_difference[i], i])


In [ ]:
sorted_mlp_list = sorted(mean_mlp_difference_arr, key=lambda x: x[0].item(), reverse = True)

In [ ]:
sorted_mlp_list

[[tensor(4.9755), 0],
 [tensor(1.5229), 10],
 [tensor(1.3259), 7],
 [tensor(1.3249), 8],
 [tensor(1.1937), 3],
 [tensor(1.1153), 11],
 [tensor(0.6438), 13],
 [tensor(0.2977), 1],
 [tensor(0.2939), 12],
 [tensor(0.2584), 4],
 [tensor(0.2380), 2],
 [tensor(0.2344), 5],
 [tensor(0.1885), 9],
 [tensor(-0.1673), 6],
 [tensor(-0.1762), 15],
 [tensor(-0.3994), 14]]

## Creating Subgraph (Threshold = 0.3) (Sparsity = 17/80 = 21%)


In [238]:
threshold = 0.3

attn_chosen = {}
for x, l, h in sorted_attention_list:
    if x.item()>threshold:
        if l not in attn_chosen:
            attn_chosen[l] = [h]
        else:
            attn_chosen[l].append(h)
            
mlp_chosen = [i for x,i in sorted_mlp_list if x.item()>threshold]

In [239]:
def make_attn_patch_hook(heads_to_corrupt, corrupt_full_output):
    def hook(module, input, output):
        for j in heads_to_corrupt:
            output[0][:, :, j*512:(j+1)*512] = corrupt_full_output[:, :, j*512:(j+1)*512]
        return output
    return hook

In [240]:
for name, layer in model.named_modules():
    layer._forward_hooks.clear()

In [241]:
len(activations['model.layers.0.mlp'])

95

In [242]:
# Sufficiency Code

subgraph_diff_suff = []

for k, data in enumerate(ds):
    hooks = []
    tokens = tokenizer(data['clean_prompt'], return_tensors='pt').to(device=device)
    a_id = tokenizer(' ' + data['name_A'], return_tensors='pt')['input_ids'][0][-1]
    b_id = tokenizer(' ' + data['name_B'], return_tensors='pt')['input_ids'][0][-1]
    
    for i in range(16):
        heads = []
        for name, layer in model.named_modules():
            if name == f'model.layers.{i}.mlp':
                if i not in mlp_chosen:
                    h = layer.register_forward_hook(
                        patching_hook_mlp(activations[name][k])
                    )
                    hooks.append(h)
            if name == f'model.layers.{i}.self_attn':
                if i in attn_chosen.keys():
                    for j in range(4):
                        if j in attn_chosen[i]:
                            continue
                        else:
                            heads.append(j)
                else:
                    for j in range(4):
                        heads.append(j)

                h = layer.register_forward_hook(make_attn_patch_hook(heads, activations[name][k]))
                hooks.append(h)

    output = model(**tokens)
    
    for h in hooks:
        h.remove()
    
    a = output.logits[:, -1, :][0][a_id]
    b = output.logits[:, -1, :][0][b_id]
    subgraph_diff_suff.append((a - b).detach().cpu())

In [243]:
len(subgraph_diff_suff)

95

In [244]:
torch.mean(torch.stack(subgraph_diff_suff))

tensor(0.4080, dtype=torch.float16)

In [ ]:
# Necessity Code and Results
subgraph_diff = []

for k, data in enumerate(ds):
    hooks = []
    tokens = tokenizer(data['clean_prompt'], return_tensors='pt').to(device=device)
    a_id = tokenizer(' ' + data['name_A'], return_tensors='pt')['input_ids'][0][-1]
    b_id = tokenizer(' ' + data['name_B'], return_tensors='pt')['input_ids'][0][-1]
    
    for i in range(16):
        heads = []
        for name, layer in model.named_modules():
            if name == f'model.layers.{i}.mlp':
                if i in mlp_chosen:
                    h = layer.register_forward_hook(
                        patching_hook_mlp(activations[name][k])
                    )
                    hooks.append(h)
            if name == f'model.layers.{i}.self_attn':
                if i in attn_chosen.keys():
                    for j in range(4):
                        if j in attn_chosen[i]:
                            heads.append(j)

                    h = layer.register_forward_hook(make_attn_patch_hook(heads, activations[name][k]))
                    hooks.append(h)
    output = model(**tokens)
    
    for h in hooks:
        h.remove()
    
    a = output.logits[:, -1, :][0][a_id]
    b = output.logits[:, -1, :][0][b_id]
    subgraph_diff.append((a - b).detach().cpu())

In [ ]:
len(subgraph_diff)

95

In [ ]:
torch.mean(torch.stack(subgraph_diff))

tensor(0.4951, dtype=torch.float16)

## Switching the important nodes based on threshold to wrong activations (corrupted), to check whether those nodes are necessary for the model, results comparison

In [245]:
import random

def generate_dict_attn(total_values=10, value_range=(0, 15)):
    result = {}
    remaining = total_values
    num_keys = random.randint(1, min(16, total_values))
    keys = random.sample(range(value_range[0], value_range[1] + 1), num_keys)

    for i, key in enumerate(keys):
        if i == num_keys - 1:
            size = remaining
        else:
            size = random.randint(1, remaining - (num_keys - i - 1))
        values = [random.randint(value_range[0], 3) for _ in range(size)]

        result[key] = values
        remaining -= size

    return result



yes_attn_random = generate_dict_attn()

In [250]:
yes_attn_random 

{14: [3, 3, 1, 3, 2, 3, 1, 1], 8: [3], 4: [3]}

In [246]:
yes_mlp_random = [random.randint(0, 15) for _ in range(7)]

In [249]:
yes_mlp_random

[2, 10, 9, 13, 8, 4, 3]

In [ ]:
# Necessity Code and Results for random baseline
subgraph_random = []

for k, data in enumerate(ds):
    hooks = []
    tokens = tokenizer(data['clean_prompt'], return_tensors='pt').to(device=device)
    a_id = tokenizer(' ' + data['name_A'], return_tensors='pt')['input_ids'][0][-1]
    b_id = tokenizer(' ' + data['name_B'], return_tensors='pt')['input_ids'][0][-1]
    
    for i in range(16):
        heads = []
        for name, layer in model.named_modules():
            if name == f'model.layers.{i}.mlp':
                if i in yes_mlp_random:
                    h = layer.register_forward_hook(
                        patching_hook_mlp(activations[name][k])
                    )
                    hooks.append(h)
            if name == f'model.layers.{i}.self_attn':
                if i in yes_attn_random.keys():
                    for j in range(4):
                        if j in yes_attn_random[i]:
                            heads.append(j)

                    h = layer.register_forward_hook(make_attn_patch_hook(heads, activations[name][k]))
                    hooks.append(h)
    output = model(**tokens)
    
    for h in hooks:
        h.remove()
    
    a = output.logits[:, -1, :][0][a_id]
    b = output.logits[:, -1, :][0][b_id]
    subgraph_random.append((a - b).detach().cpu())

In [251]:
torch.mean(torch.stack(subgraph_random))

tensor(0.6655, dtype=torch.float16)

# Complete Token By Token Patching

Token by token patching means, for every token, at every layer, we patch and see the results.

In [212]:
for name, layer in model.named_modules():
    layer._forward_hooks.clear()

In [213]:
activations_complete = {}

def find_hook(i, name):
    def hook(model, input, output):
        data = output[0]
        data = data.detach().cpu()
        if i not in activations_complete:
            activations_complete[i] = {}
        else:
            print(i)
        if name not in activations_complete[i]:
            activations_complete[i][name] = []
        if 'mlp' in name:
            activations_complete[i][name].append(data[i, :])
        else:
            activations_complete[i][name].append(data[:, i, :])

    return hook

In [214]:
for k, data in enumerate(ds):
    tokens = tokenizer(data['corrupted_prompt'], return_tensors = 'pt').to(device = device)
    a_id = tokenizer(' ' + data['name_A'], return_tensors = 'pt')['input_ids'][0][-1]
    b_id = tokenizer(' ' + data['name_B'], return_tensors = 'pt')['input_ids'][0][-1]
    for i in range(16): #seq_len
        for j in range(16): #layer
            for name, layer in model.named_modules():
                if name==f'model.layers.{j}.mlp':
                    h = layer.register_forward_hook(find_hook(i, name))
                    output = model(**tokens)
                    h.remove()
                if name==f'model.layers.{j}.self_attn':
                    h = layer.register_forward_hook(find_hook(i, name))
                    output = model(**tokens)
                    h.remove()


0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
3
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
5
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
6
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
7
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
8
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
9
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
10
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
11
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
12
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
13
14
14
14

In [224]:
activations_complete[0]['model.layers.1.self_attn'][0].shape

torch.Size([1, 2048])

## MLP Patching (Complete)

In [262]:
for name, layer in model.named_modules():
    layer._forward_hooks.clear()

In [263]:
def mlp_complete_hook(token, corrupt):
    def hook(model, input, output):
        output[0][token, :] = corrupt.to(output.device)

        return output
    return hook

In [264]:
store_patch_complete_mlp = torch.zeros([95, 16, 16])


for k, data in enumerate(ds):
    tokens = tokenizer(data['clean_prompt'], return_tensors = 'pt').to(device = device)
    a_id = tokenizer(' ' + data['name_A'], return_tensors = 'pt')['input_ids'][0][-1]
    b_id = tokenizer(' ' + data['name_B'], return_tensors = 'pt')['input_ids'][0][-1]
    for token in range(16): #seq_len
        for i in range(16): #layers
            for name, layer in model.named_modules():
                if name==f'model.layers.{i}.mlp':
                    h = layer.register_forward_hook(mlp_complete_hook(token, activations_complete[token][name][k]))
                    output = model(**tokens)
                    h.remove()
                    a = output.logits[:,-1,:][0][a_id]
                    b = output.logits[:,-1,:][0][b_id]
                    c = a-b
                    d = c.detach().to(device = 'cpu')
                    float(d)
                    store_patch_complete_mlp[k, token, i] = d

In [267]:
mean_patch_complete_mlp = torch.mean(store_patch_complete_mlp, axis = 0)

In [ ]:
mean_complete_mlp_diff = correct_mean - mean_patch_complete_mlp

In [297]:
mean_patch_complete_mlp_arr = []

for i in range(len(mean_complete_mlp_diff)):
    for j in range(len(mean_complete_mlp_diff[i])):
        mean_patch_complete_mlp_arr.append([mean_complete_mlp_diff[i,j], i, j])

In [298]:
sorted_mlp_complete_list = sorted(mean_patch_complete_mlp_arr, key=lambda x: x[0].item(), reverse = True)

In [299]:
sorted_mlp_complete_list

[[tensor(2.8528), 11, 0],
 [tensor(2.1233), 12, 0],
 [tensor(0.8829), 11, 7],
 [tensor(0.7002), 15, 11],
 [tensor(0.6681), 11, 8],
 [tensor(0.6336), 15, 10],
 [tensor(0.6194), 11, 3],
 [tensor(0.5657), 12, 8],
 [tensor(0.5292), 12, 3],
 [tensor(0.4625), 12, 7],
 [tensor(0.4563), 12, 10],
 [tensor(0.3690), 11, 10],
 [tensor(0.2736), 15, 13],
 [tensor(0.2394), 12, 11],
 [tensor(0.2335), 11, 9],
 [tensor(0.2320), 11, 11],
 [tensor(0.2142), 11, 13],
 [tensor(0.1845), 12, 9],
 [tensor(0.1806), 11, 1],
 [tensor(0.1805), 12, 13],
 [tensor(0.1620), 11, 5],
 [tensor(0.1378), 11, 12],
 [tensor(0.1368), 12, 4],
 [tensor(0.1314), 15, 8],
 [tensor(0.1250), 12, 2],
 [tensor(0.1168), 11, 2],
 [tensor(0.1075), 11, 14],
 [tensor(0.1058), 12, 1],
 [tensor(0.0935), 15, 12],
 [tensor(0.0927), 15, 4],
 [tensor(0.0703), 12, 5],
 [tensor(0.0641), 12, 12],
 [tensor(0.0538), 11, 4],
 [tensor(0.0513), 12, 14],
 [tensor(0.0301), 13, 10],
 [tensor(0.0269), 12, 6],
 [tensor(0.0214), 13, 3],
 [tensor(0.0211), 15, 5

## ATTENTION Patching (Complete)

In [300]:
for name, layer in model.named_modules():
    layer._forward_hooks.clear()

In [301]:
def patching_hook_complete_attention(in_head, out_head , corrupt, token):
    def hook(model, input, output):
        output[0][:,token,in_head : out_head] = corrupt
        return output
    return hook

In [303]:
store_patch_complete_attn = torch.zeros([95, 16, 16, 4])


for k, data in enumerate(ds):
    tokens = tokenizer(data['clean_prompt'], return_tensors = 'pt').to(device = device)
    token_len = tokens['input_ids'].shape[1]
    #print(tokens['input_ids'].shape)
    a_id = tokenizer(' ' + data['name_A'], return_tensors = 'pt')['input_ids'][0][-1]
    b_id = tokenizer(' ' + data['name_B'], return_tensors = 'pt')['input_ids'][0][-1]
    for token in range(token_len):
        for i in range(16):
            temp[token][i] = {}
            for j in range(4):
                for name, layer in model.named_modules():
                    if name==f'model.layers.{i}.self_attn':
                        h = layer.register_forward_hook(patching_hook_complete_attention(j*512, (j+1)*512, activations_complete[token][name][k][:, j*512: (j+1)*512], token))
                        output = model(**tokens)
                        h.remove()
                        a = output.logits[:,-1,:][0][a_id]
                        b = output.logits[:,-1,:][0][b_id]
                        c = a-b
                        d = c.detach().to(device = 'cpu')
                        float(d)
                        store_patch_complete_attn[k, token, i, j] = d

In [309]:
store_patch_complete_attn[5][6][6]

tensor([3.5625, 3.5625, 3.5625, 3.5625])

In [310]:
mean_store_patch_complete = torch.mean(store_patch_complete_attn, axis = 0)

In [311]:
mean_store_complete_diff = correct_mean - mean_store_patch_complete

In [ ]:
mean_store_complete_diff_arr = []
for i in range(len(mean_store_complete_diff)):
    for j in range(len(mean_store_complete_diff[i])):
        for k in range(len(mean_store_complete_diff[i][j])):
            mean_store_complete_diff_arr.append([mean_store_complete_diff[i][j][k], i, j, k]) #i - Token, j - layer, k - head



In [316]:
sorted_attn_complete_list = sorted(mean_store_complete_diff_arr, key=lambda x: x[0].item(), reverse = True)

In [317]:
sorted_attn_complete_list

[[tensor(0.5519), 15, 14, 2],
 [tensor(0.5464), 15, 14, 0],
 [tensor(0.5108), 15, 14, 3],
 [tensor(0.5055), 15, 9, 3],
 [tensor(0.5047), 15, 14, 1],
 [tensor(0.4299), 15, 9, 2],
 [tensor(0.3820), 11, 8, 2],
 [tensor(0.3394), 15, 15, 2],
 [tensor(0.3142), 15, 9, 1],
 [tensor(0.3009), 15, 11, 0],
 [tensor(0.2819), 15, 11, 2],
 [tensor(0.2775), 15, 15, 3],
 [tensor(0.2726), 12, 8, 2],
 [tensor(0.2624), 15, 11, 1],
 [tensor(0.2490), 15, 15, 1],
 [tensor(0.2461), 15, 11, 3],
 [tensor(0.2429), 15, 15, 0],
 [tensor(0.1552), 15, 8, 3],
 [tensor(0.1293), 12, 8, 0],
 [tensor(0.1285), 15, 8, 0],
 [tensor(0.1158), 11, 8, 0],
 [tensor(0.0988), 15, 9, 0],
 [tensor(0.0907), 15, 8, 2],
 [tensor(0.0718), 11, 12, 2],
 [tensor(0.0694), 12, 7, 2],
 [tensor(0.0657), 15, 8, 1],
 [tensor(0.0641), 15, 6, 1],
 [tensor(0.0622), 15, 13, 3],
 [tensor(0.0611), 11, 7, 2],
 [tensor(0.0561), 12, 5, 2],
 [tensor(0.0556), 12, 8, 1],
 [tensor(0.0502), 15, 13, 1],
 [tensor(0.0480), 12, 12, 2],
 [tensor(0.0477), 11, 14, 3

## Threshold and Node Selection

# Final Results (Threshold 0.1)

In [319]:
threshold = 0.1

In [320]:
mlp_complete_chosen_list = {}
for x, token, layer in sorted_mlp_complete_list:
    if x.item()>threshold:
        if token not in mlp_complete_chosen_list:
            mlp_complete_chosen_list[token] = [layer]
        else:
            mlp_complete_chosen_list[token].append(layer)


In [321]:
mlp_complete_chosen_list

{11: [0, 7, 8, 3, 10, 9, 11, 13, 1, 5, 12, 2, 14],
 12: [0, 8, 3, 7, 10, 11, 9, 13, 4, 2, 1],
 15: [11, 10, 13, 8]}

In [323]:
attn_complete_chosen_list = {}
for x, token, layer, head in sorted_attn_complete_list:
    if x.item()>threshold:
        if token not in attn_complete_chosen_list:
            attn_complete_chosen_list[token] = {}
        if layer not in attn_complete_chosen_list[token]:
            attn_complete_chosen_list[token][layer] = [head]
        else:
            attn_complete_chosen_list[token][layer].append(head)

In [365]:
attn_complete_chosen_list

{15: {14: [2, 0, 3, 1],
  9: [3, 2, 1],
  15: [2, 3, 1, 0],
  11: [0, 2, 1, 3],
  8: [3, 0]},
 11: {8: [2, 0]},
 12: {8: [2, 0]}}

In [353]:
for name, layer in model.named_modules():
    layer._forward_hooks.clear()

In [354]:
def mlp_complete_correct_hook(token, corrupt):
    def hook(model, input, output):
        output[0][token, :] = corrupt
        return output
    return hook

In [355]:
def correct_hook_complete_attention(heads, corrupt, token):
    def hook(model, input, output):
        for head_idx in heads:
            output[0][:,token, head_idx*512 : (head_idx+1)*512] = corrupt[:, head_idx*512 : (head_idx+1)*512]

        return output
    return hook

In [357]:
# sufficiency

store_correct_corrupted = []

for k, data in enumerate(ds):

    print(k)
    hooks = []
    heads = []
    tokens = tokenizer(data['clean_prompt'], return_tensors = 'pt').to(device = device)
    token_len = tokens['input_ids'].shape[1]
    a_id = tokenizer(' ' + data['name_A'], return_tensors = 'pt')['input_ids'][0][-1]
    b_id = tokenizer(' ' + data['name_B'], return_tensors = 'pt')['input_ids'][0][-1]
    for name, layer in model.named_modules():
        for token in range(16):
            for i in range(16):
                if name==f'model.layers.{i}.self_attn':
                    if token in attn_complete_chosen_list:
                        if i in attn_complete_chosen_list[token]:
                            for j in range(4):
                                if j in attn_complete_chosen_list[token][i]:
                                    continue
                                else:
                                    heads.append(j)
                            hooks.append(layer.register_forward_hook(correct_hook_complete_attention(heads, activations_complete[token][name][k], token)))
                        else:
                            for j in range(4):
                                heads.append(j)
                            hooks.append(layer.register_forward_hook(correct_hook_complete_attention(heads, activations_complete[token][name][k], token)))
                    else:
                        for j in range(4):
                            heads.append(j)
                        hooks.append(layer.register_forward_hook(correct_hook_complete_attention(heads, activations_complete[token][name][k], token)))
                if name==f'model.layers.{i}.mlp':
                    if token in mlp_complete_chosen_list:
                        if i in mlp_complete_chosen_list[token]:
                            continue
                        else:
                            hooks.append(layer.register_forward_hook(mlp_complete_correct_hook(token , activations_complete[token][name][k])))
                    else:
                        hooks.append(layer.register_forward_hook(mlp_complete_correct_hook(token , activations_complete[token][name][k])))
    output = model(**tokens) #gives logits
    for h in hooks:
        h.remove()
    a = output.logits[:,-1,:][0][a_id]
    b = output.logits[:,-1,:][0][b_id]
    c = a-b
    d = c.detach().to(device = 'cpu')
    float(d)
    store_correct_corrupted.append(d)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94


In [359]:
corrupted_correct_mean = torch.mean(torch.stack(store_correct_corrupted))

In [360]:
corrupted_correct_mean

tensor(0.5337, dtype=torch.float16)

In [361]:
correct_mean

tensor(5.3867, dtype=torch.float16)

In [377]:
# necessity

store_correct_corrupted_nec = []

for k, data in enumerate(ds):

    print(k)
    hooks = []
    heads = []
    tokens = tokenizer(data['clean_prompt'], return_tensors = 'pt').to(device = device)
    token_len = tokens['input_ids'].shape[1]
    a_id = tokenizer(' ' + data['name_A'], return_tensors = 'pt')['input_ids'][0][-1]
    b_id = tokenizer(' ' + data['name_B'], return_tensors = 'pt')['input_ids'][0][-1]
    for name, layer in model.named_modules():
        for token in range(16):
            for i in range(16):
                if name==f'model.layers.{i}.self_attn':
                    if token in attn_complete_chosen_list:
                        if i in attn_complete_chosen_list[token]:
                            for j in range(4):
                                if j in attn_complete_chosen_list[token][i]:
                                    heads.append(j)
                            hooks.append(layer.register_forward_hook(correct_hook_complete_attention(heads, activations_complete[token][name][k], token)))
                if name==f'model.layers.{i}.mlp':
                    if token in mlp_complete_chosen_list:
                        if i in mlp_complete_chosen_list[token]:
                            hooks.append(layer.register_forward_hook(mlp_complete_correct_hook(token , activations_complete[token][name][k])))
    output = model(**tokens) #gives logits
    for h in hooks:
        h.remove()
    a = output.logits[:,-1,:][0][a_id]
    b = output.logits[:,-1,:][0][b_id]
    c = a-b
    d = c.detach().to(device = 'cpu')
    float(d)
    store_correct_corrupted_nec.append(d)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94


In [378]:
corrupted_correct_mean_nec = torch.mean(torch.stack(store_correct_corrupted_nec))

In [379]:
corrupted_correct_mean_nec

tensor(0.5210, dtype=torch.float16)

## Random Baseline Run

In [366]:
import random

def generate_dict_mlp(total_values=28, value_range=(0, 15)):
    result = {}
    remaining = total_values
    num_keys = random.randint(1, min(16, total_values))
    keys = random.sample(range(value_range[0], value_range[1] + 1), num_keys)

    for i, key in enumerate(keys):
        if i == num_keys - 1:
            size = remaining
        else:
            size = random.randint(1, remaining - (num_keys - i - 1))
        values = [random.randint(value_range[0], value_range[1]) for _ in range(size)]

        result[key] = values
        remaining -= size

    return result



yes_mlp_random = generate_dict_mlp()
print(yes_mlp_random)

{8: [7], 7: [5], 9: [12, 7, 0, 6, 14], 0: [5, 14, 7, 7, 0, 15, 7, 10, 12, 13, 13, 12, 11], 6: [15], 10: [9, 6, 11, 15], 12: [2], 5: [14], 4: [7]}


In [367]:
import random

def generate_nested_dict(total_values=21, value_range=(0, 15)):
    result = {}
    outer_key = random.randint(value_range[0], value_range[1])

    inner_dict = {}
    remaining = total_values
    num_inner_keys = random.randint(1, min(16, total_values))
    inner_keys = random.sample(range(value_range[0], value_range[1] + 1), num_inner_keys)

    for i, key in enumerate(inner_keys):
        if i == num_inner_keys - 1:
            size = remaining
        else:
            size = random.randint(1, remaining - (num_inner_keys - i - 1))
        values = [random.randint(value_range[0], value_range[1]) for _ in range(size)]

        inner_dict[key] = values
        remaining -= size

    result[outer_key] = inner_dict
    return result
yes_attn_random = generate_nested_dict()
print(yes_attn_random)

{14: {15: [1, 3, 10, 12, 12, 13, 12, 4, 13], 9: [10, 15, 13, 3, 4, 10], 5: [8, 4], 8: [10, 8], 4: [9], 13: [9]}}


In [368]:
for name, layer in model.named_modules():
    layer._forward_hooks.clear()

In [370]:
# necessity

store_correct_corrupted_nec_random = []

for k, data in enumerate(ds):

    print(k)
    hooks = []
    heads = []
    tokens = tokenizer(data['clean_prompt'], return_tensors = 'pt').to(device = device)
    token_len = tokens['input_ids'].shape[1]
    a_id = tokenizer(' ' + data['name_A'], return_tensors = 'pt')['input_ids'][0][-1]
    b_id = tokenizer(' ' + data['name_B'], return_tensors = 'pt')['input_ids'][0][-1]
    for name, layer in model.named_modules():
        for token in range(16):
            for i in range(16):
                if name==f'model.layers.{i}.self_attn':
                    if token in yes_attn_random:
                        if i in yes_attn_random[token]:
                            for j in range(4):
                                if j in yes_attn_random[token][i]:
                                    heads.append(j)
                            hooks.append(layer.register_forward_hook(correct_hook_complete_attention(heads, activations_complete[token][name][k], token)))
                if name==f'model.layers.{i}.mlp':
                    if token in yes_mlp_random:
                        if i in yes_mlp_random[token]:
                            hooks.append(layer.register_forward_hook(mlp_complete_correct_hook(token , activations_complete[token][name][k])))
    output = model(**tokens) #gives logits
    for h in hooks:
        h.remove()
    a = output.logits[:,-1,:][0][a_id]
    b = output.logits[:,-1,:][0][b_id]
    c = a-b
    d = c.detach().to(device = 'cpu')
    float(d)
    store_correct_corrupted_nec_random.append(d)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94


In [371]:
corrupted_correct_mean_nec_random = torch.mean(torch.stack(store_correct_corrupted_nec_random))

In [373]:
corrupted_correct_mean_nec_random

tensor(5.2227, dtype=torch.float16)